<a href="https://www.kaggle.com/code/kmljts/analyzing-2015-us-flight-delays-with-spark?scriptVersionId=265776332" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os

In [2]:
airports_path, airlines_path, flights_path = [os.path.join(dirname, filename) for dirname, _, filenames in os.walk('/kaggle/input') for filename in filenames]

airports_path, airlines_path, flights_path

('/kaggle/input/flight-delays/airports.csv',
 '/kaggle/input/flight-delays/airlines.csv',
 '/kaggle/input/flight-delays/flights.csv')

In [3]:
spark = SparkSession.builder \
    .appName("AirlineDelayAnalysis") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/04 23:38:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Flights

In [4]:
flights = spark.read.csv(flights_path, header=True, inferSchema=True)

flights

DataFrame[YEAR: int, MONTH: int, DAY: int, DAY_OF_WEEK: int, AIRLINE: string, FLIGHT_NUMBER: int, TAIL_NUMBER: string, ORIGIN_AIRPORT: string, DESTINATION_AIRPORT: string, SCHEDULED_DEPARTURE: int, DEPARTURE_TIME: int, DEPARTURE_DELAY: int, TAXI_OUT: int, WHEELS_OFF: int, SCHEDULED_TIME: int, ELAPSED_TIME: int, AIR_TIME: int, DISTANCE: int, WHEELS_ON: int, TAXI_IN: int, SCHEDULED_ARRIVAL: int, ARRIVAL_TIME: int, ARRIVAL_DELAY: int, DIVERTED: int, CANCELLED: int, CANCELLATION_REASON: string, AIR_SYSTEM_DELAY: int, SECURITY_DELAY: int, AIRLINE_DELAY: int, LATE_AIRCRAFT_DELAY: int, WEATHER_DELAY: int]

In [5]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [6]:
flights.show(5)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

## Airports

In [7]:
airports = spark.read.csv(airports_path, header=True, inferSchema=True)

airports

DataFrame[IATA_CODE: string, AIRPORT: string, CITY: string, STATE: string, COUNTRY: string, LATITUDE: double, LONGITUDE: double]

In [8]:
airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [9]:
airports.show(5)

+---------+--------------------+-----------+-----+-------+--------+----------+
|IATA_CODE|             AIRPORT|       CITY|STATE|COUNTRY|LATITUDE| LONGITUDE|
+---------+--------------------+-----------+-----+-------+--------+----------+
|      ABE|Lehigh Valley Int...|  Allentown|   PA|    USA|40.65236|  -75.4404|
|      ABI|Abilene Regional ...|    Abilene|   TX|    USA|32.41132|  -99.6819|
|      ABQ|Albuquerque Inter...|Albuquerque|   NM|    USA|35.04022|-106.60919|
|      ABR|Aberdeen Regional...|   Aberdeen|   SD|    USA|45.44906| -98.42183|
|      ABY|Southwest Georgia...|     Albany|   GA|    USA|31.53552| -84.19447|
+---------+--------------------+-----------+-----+-------+--------+----------+
only showing top 5 rows



## Airlines

In [10]:
airlines = spark.read.csv(airlines_path, header=True, inferSchema=True)

airlines

DataFrame[IATA_CODE: string, AIRLINE: string]

In [11]:
airlines.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRLINE: string (nullable = true)



In [12]:
airlines.show(5)

+---------+--------------------+
|IATA_CODE|             AIRLINE|
+---------+--------------------+
|       UA|United Air Lines ...|
|       AA|American Airlines...|
|       US|     US Airways Inc.|
|       F9|Frontier Airlines...|
|       B6|     JetBlue Airways|
+---------+--------------------+
only showing top 5 rows



## EDA

In [13]:
busiest_airports = flights.groupBy('ORIGIN_AIRPORT').agg(F.count('*').alias('origin_count'))

busiest_airports.orderBy(F.col('origin_count').desc()).show(10)

+--------------+------------+
|ORIGIN_AIRPORT|origin_count|
+--------------+------------+
|           ATL|      346836|
|           ORD|      285884|
|           DFW|      239551|
|           DEN|      196055|
|           LAX|      194673|
|           SFO|      148008|
|           PHX|      146815|
|           IAH|      146622|
|           LAS|      133181|
|           MSP|      112117|
+--------------+------------+
only showing top 10 rows



In [14]:
busiest_airports_full = busiest_airports.join(airports, busiest_airports.ORIGIN_AIRPORT == airports.IATA_CODE, how='inner')

busiest_airports_full.orderBy(F.col('origin_count').desc()).select(airports.IATA_CODE, airports.AIRPORT, busiest_airports.origin_count).show(10)

+---------+--------------------+------------+
|IATA_CODE|             AIRPORT|origin_count|
+---------+--------------------+------------+
|      ATL|Hartsfield-Jackso...|      346836|
|      ORD|Chicago O'Hare In...|      285884|
|      DFW|Dallas/Fort Worth...|      239551|
|      DEN|Denver Internatio...|      196055|
|      LAX|Los Angeles Inter...|      194673|
|      SFO|San Francisco Int...|      148008|
|      PHX|Phoenix Sky Harbo...|      146815|
|      IAH|George Bush Inter...|      146622|
|      LAS|McCarran Internat...|      133181|
|      MSP|Minneapolis-Saint...|      112117|
+---------+--------------------+------------+
only showing top 10 rows



In [15]:
flights.groupBy('AIRLINE').agg(F.avg('ARRIVAL_DELAY').alias('avg_delay')).orderBy(F.col('avg_delay').desc()).show()

+-------+-------------------+
|AIRLINE|          avg_delay|
+-------+-------------------+
|     NK| 14.471799501705833|
|     F9| 12.504706404706404|
|     B6|  6.677860800940307|
|     EV|  6.585378691739733|
|     MQ|  6.457873460764516|
|     OO|  5.845652151300072|
|     UA|  5.431593935741549|
|     VX|  4.737705721003135|
|     WN| 4.3749636792570525|
|     US| 3.7062088424131026|
|     AA| 3.4513721447256764|
|     HA|  2.023092805197196|
|     DL|0.18675361236390797|
|     AS|-0.9765630924118783|
+-------+-------------------+

